# 무라벨 삭제 집합 D — 전체 파이프라인 원응답 수집 (제출용 아님)

v3·v17 삭제 후보(`experiments/quoted_deletion_candidate.py`)가 무라벨에서 **무엇을 지우는지**를 보려고
운영 `script.py` 를 무라벨 표본에 그대로 돌립니다. 모델 입력·프롬프트·설정은 제출 코드와 같습니다.
D 계산과 라벨 추첨은 PC 에서 합니다. 이 노트북은 추론만 합니다.

1. 런타임을 **A100 GPU** 로 설정합니다. VRAM 30GiB·여유 디스크 80GiB 를 사전 검사합니다.
2. `MyDrive/a5/train_unlabeled.jsonl` 이 있어야 합니다(A5 때 올린 파일, 20,000건·SHA256 확인).
3. Colab 보안 비밀 `HF_TOKEN` 을 등록하고 노트북 접근을 허용합니다.
4. 위에서부터 모두 실행합니다. 기본은 seed 20260923 순서의 앞 **6,000건**을 500건씩 12묶음으로 돌립니다. 회차 1 에서 끝난 u00~u03 은 건너뛰므로 실제로는 8묶음입니다.
5. 묶음이 끝날 때마다 Drive `MyDrive/a5/unlabeled-d-<코드 SHA 12자리>/` 에 저장됩니다.
   런타임이 끊기면 **첫 셀부터 다시 실행**하세요. 끝난 묶음은 건너뜁니다.
6. 끝나면 마지막 셀의 ZIP 을 받습니다. 실패했어도 마지막 셀은 실행합니다.

예상 시간: 설치·모델 준비 약 15분 + 묶음당 약 34분(회차 1 실측 건당 3.7초) ≈ 새 8묶음 **약 4시간 40분**.
같은 Drive 폴더를 두 런타임에서 동시에 돌리지 마세요.

In [ ]:
import csv
import gzip
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

WORK = Path(tempfile.mkdtemp(prefix="unlabeled-d-", dir="/content"))
RESULTS = WORK / "results"
RESULTS.mkdir()
PYTHON = str(WORK / "venv/bin/python")
MODEL_ID = "google/gemma-4-26B-A4B-it"
REVISION = "4d7ae4984b7db7de8f8457170b3f1a419ee76d52"
EXPECTED_PACKAGES = {"vllm": "0.26.0", "torch": "2.11.0+cu130",
                     "transformers": "5.14.1", "xgrammar": "0.2.3"}
SERVER_PYTHON = "3.12.13"
case_inputs = {}
REPO_URL = "https://github.com/LittleBitAI/ai-nara-shop.git"
REPO_REF = "14f03d1d5426920cdb9ca7af55e482a3ed1c777e"  # 표본 ID 순서를 고정한 커밋. 변경하지 않음
N_NOTICES = 6000   # seed 순서의 앞 몇 건. 회차 1(2,000건)에서 D 가 모자라 6,000 으로 늘렸습니다. 끝난 묶음은 건너뜁니다
SHARD_SIZE = 500   # 한 번의 script.py 실행 단위. 끊겨도 이 단위로 이어집니다

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n",
                    encoding="utf-8", newline="\n")

def run_logged(name, command, env=None, cwd=None):
    started = time.time()
    entry = {"argv": [str(x) for x in command], "cwd": str(cwd or WORK), "started": started,
             "returncode": None}
    if entry["argv"][0] == PYTHON:
        # Absolute Python paths do not activate PATH for tools such as ninja.
        env = dict(os.environ if env is None else env)
        env["PATH"] = str(Path(PYTHON).parent) + os.pathsep + env.get("PATH", "")
        env["VIRTUAL_ENV"] = str(Path(PYTHON).parent.parent)
    log_path = RESULTS / (name + ".log")
    record_path = RESULTS / (name + "-command.json")
    if log_path.exists() or record_path.exists():
        raise ValueError(f"{name} 실행 기록이 이미 있습니다. 첫 셀부터 새 작업 폴더로 실행하세요.")
    write_json(record_path, entry)
    try:
        with log_path.open("x", encoding="utf-8", newline="\n") as log_file:
            with subprocess.Popen(entry["argv"], cwd=entry["cwd"], env=env, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                  errors="replace", bufsize=1) as process:
                try:
                    for line in process.stdout:
                        log_file.write(line)
                        log_file.flush()
                        print(line, end="")
                    entry["returncode"] = process.wait()
                except BaseException:
                    process.terminate()
                    try:
                        process.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        process.kill()
                    raise
    finally:
        entry["elapsed_seconds"] = time.time() - started
        write_json(record_path, entry)
    if entry["returncode"] != 0:
        raise RuntimeError(f"{name} 실패 (exit={entry['returncode']}). 마지막 로그 다운로드 셀을 실행하세요.")
    return log_path

write_json(RESULTS / "host.json", {"python": sys.version, "work": str(WORK)})
print("작업 폴더:", WORK)

## 코드 고정

In [ ]:
REPO = WORK / "repo"
run_logged("git-clone", ["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
run_logged("git-fetch", ["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO)
run_logged("git-checkout", ["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if SOURCE_COMMIT != REPO_REF:
    raise ValueError("REPO_REF를 실행 안내의 40자리 SHA로 고정하세요.")
SUBMISSION = REPO  # 고정 환경 설치 셀이 requirements.txt 경로로 사용
write_json(RESULTS / "source.json", {"commit": SOURCE_COMMIT, "requested_ref": REPO_REF})
IDS_FILE = REPO / "reports/label-compare/unlabeled-d/ids.txt"
ids_manifest = json.loads(IDS_FILE.with_suffix(".manifest.json").read_text(encoding="utf-8"))
if hashlib.sha256(IDS_FILE.read_bytes()).hexdigest() != ids_manifest["ids_sha256"]:
    raise ValueError("표본 ID 파일 해시가 manifest 와 다릅니다")
ORDER = IDS_FILE.read_text(encoding="utf-8").split()
if not 0 < N_NOTICES <= len(ORDER):
    raise ValueError(f"N_NOTICES 는 1~{len(ORDER)} 입니다")
print("고정 코드:", SOURCE_COMMIT, "표본:", N_NOTICES, "건")

## Drive 입력 확인 — 업로드 후 이 셀부터 재실행 가능

A5 때 올린 `MyDrive/a5/train_unlabeled.jsonl` 을 그대로 씁니다. 없으면 PC 의 `open/train_unlabeled.jsonl`(약 791MB)을
Google Drive 웹 화면에서 **내 드라이브 → a5 폴더**에 올리고 이 셀부터 다시 실행하세요.
다른 위치에 있으면 `UNLABELED_PATH` 만 바꿉니다. 표본 공고만 뽑아 500건씩 묶음 파일을 만듭니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
UNLABELED_PATH = Path("/content/drive/MyDrive/a5/train_unlabeled.jsonl")
OUT_ROOT = Path("/content/drive/MyDrive/a5/unlabeled-d-" + SOURCE_COMMIT[:12])
if not UNLABELED_PATH.is_file():
    write_json(RESULTS / "input-check.json", {"status": "missing", "path": str(UNLABELED_PATH)})
    raise FileNotFoundError(f"입력 파일이 없습니다: {UNLABELED_PATH}\nPC의 open/train_unlabeled.jsonl을 Drive의 내 드라이브/a5 폴더에 업로드한 뒤 이 셀만 다시 실행하세요.")
raw = UNLABELED_PATH.read_bytes()
input_hash = hashlib.sha256(raw).hexdigest()
if input_hash != ids_manifest["input_sha256"]:
    raise ValueError("원본 입력 SHA256 불일치 — 표본을 뽑은 파일과 다릅니다")
# JSONL 행 경계는 줄바꿈뿐입니다. splitlines() 는 본문 안의 U+2028 에서도 끊습니다.
lines = {json.loads(line)["id"]: line for line in raw.decode("utf-8").split("\n") if line.strip()}
del raw
if len(lines) != 20000:
    raise ValueError(f"무라벨 건수 {len(lines)} != 20000")
chosen = ORDER[:N_NOTICES]
SHARDS = [(f"u{k:02d}", chosen[start:start + SHARD_SIZE])
          for k, start in enumerate(range(0, len(chosen), SHARD_SIZE))]
OUT_ROOT.mkdir(parents=True, exist_ok=True)
write_json(RESULTS / "input-check.json", {"status": "verified", "path": str(UNLABELED_PATH),
    "sha256": input_hash, "count": len(lines), "notices": N_NOTICES,
    "shards": [[name, len(ids)] for name, ids in SHARDS], "out": str(OUT_ROOT)})
print("저장/재개 위치:", OUT_ROOT)
print("묶음:", [(name, len(ids)) for name, ids in SHARDS])

## 2. 실제 GPU와 디스크 확인

GPU 종류·드라이버·VRAM·RAM을 기록합니다. 30GiB VRAM/80GiB 여유 디스크는 이 노트북의 사전 거름 기준이며 적재 보장이 아닙니다. 부족하면 모델 다운로드 전에 멈춥니다.

In [ ]:
gpu_log = run_logged("gpu", ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                             "--format=csv,noheader"])
gpus = gpu_log.read_text(encoding="utf-8").strip().splitlines()
if not gpus or max(int(row.split(",")[1].strip().split()[0]) for row in gpus) < 30 * 1024:
    raise RuntimeError("VRAM 30GiB 미만입니다. 더 큰 GPU 런타임을 선택한 뒤 처음부터 실행하세요.")
resources = {"gpus": gpus, "disk_free_bytes": shutil.disk_usage(WORK).free,
             "meminfo": Path("/proc/meminfo").read_text()}
write_json(RESULTS / "resources.json", resources)
if resources["disk_free_bytes"] < 80 * 1024**3:
    raise RuntimeError("설치와 원본 모델을 준비할 여유 디스크 80GiB가 필요합니다.")
print("GPU 사전 확인 통과. 실제 적재/실행 성공은 아직 확인하지 않았습니다.")

## 고정 Python·추론 패키지 설치

기존 Colab 검증과 같은 Python 3.12.13·vLLM 0.26.0·CUDA 13.0을 사용합니다.

In [ ]:
# Colab 커널 Python과 분리해 대회 서버의 정확한 Python 버전을 준비합니다.
run_logged("uv-install", [sys.executable, "-m", "pip", "install", "uv"])
run_logged("python-install", [sys.executable, "-m", "uv", "venv", "--python", SERVER_PYTHON,
                             "--seed", str(WORK / "venv")])
run_logged("install", [PYTHON, "-m", "pip", "install",
    "--extra-index-url", "https://download.pytorch.org/whl/cu130",
    *[f"{name}=={version}" for name, version in EXPECTED_PACKAGES.items()]])
# 같은 저장소의 requirements.txt를 설치합니다.
run_logged("submission-install", [PYTHON, "-m", "pip", "install", "-r",
                                  str(SUBMISSION / "requirements.txt")])
if json.loads((RESULTS / "submission-install-command.json").read_text())["elapsed_seconds"] > 600:
    raise RuntimeError("제출 requirements 설치가 서버 제한 600초를 넘었습니다.")
run_logged("pip-freeze", [PYTHON, "-m", "pip", "freeze"])
runtime_code = """
import json, platform, shutil, subprocess, sys, torch, vllm, transformers, xgrammar
from importlib.metadata import version
from pathlib import Path
runtime = {"python": platform.python_version(), "cuda": torch.version.cuda,
           "packages": {n: version(n) for n in ["torch", "vllm", "transformers", "xgrammar"]},
           "gpu": torch.cuda.get_device_name(0),
           "ninja_path": shutil.which("ninja"),
           "ninja_version": subprocess.check_output(["ninja", "--version"], text=True).strip()}
Path(sys.argv[1]).write_text(json.dumps(runtime, indent=2) + "\\n", encoding="utf-8")
print(runtime)
torch.empty(1, device="cuda")
"""
run_logged("runtime", [PYTHON, "-c", runtime_code, str(RESULTS / "runtime.json")])
runtime = json.loads((RESULTS / "runtime.json").read_text())
if (runtime["python"] != SERVER_PYTHON or runtime["packages"] != EXPECTED_PACKAGES
        or runtime["cuda"] != "13.0"):
    raise RuntimeError("서버의 Python/핵심 패키지/CUDA 빌드와 다릅니다. 검증을 중단합니다.")

## 4. HF_TOKEN으로 고정 리비전 모델 다운로드

Colab 보안 비밀에 `HF_TOKEN`을 등록하고 이 노트북의 접근을 허용하세요. 읽지 못하면 숨김 입력으로 받으며 **빈 토큰은 거부**합니다.
토큰은 모델 다운로드 자식 프로세스 환경변수로만 전달하고, 추론 환경·명령행·로그·결과 ZIP에 기록하지 않습니다.

[고정 모델](https://huggingface.co/google/gemma-4-26B-A4B-it/tree/4d7ae4984b7db7de8f8457170b3f1a419ee76d52)의 접근 권한이 있는 계정을 사용합니다.
가중치는 준비 단계에서만 내려받고 제출물에 포함하지 않습니다.

In [ ]:
from google.colab import userdata
from getpass import getpass

try:
    token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    token = getpass("HF_TOKEN (Hugging Face 읽기 토큰, 필수): ")
if not token or not token.strip():
    raise ValueError("HF_TOKEN이 필요합니다. Colab 보안 비밀에 등록하고 노트북 접근을 허용하세요.")
download_env = {**os.environ, "HF_TOKEN": token.strip()}
download_env.pop("HF_HUB_OFFLINE", None)
download_env.pop("TRANSFORMERS_OFFLINE", None)
download_code = """
from huggingface_hub import snapshot_download
from pathlib import Path
import os
import sys
model_dir = snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], token=os.environ["HF_TOKEN"],
    allow_patterns=["*.safetensors", "*.json", "*.jinja", "*.model"])
if Path(model_dir).name != sys.argv[2]:
    raise RuntimeError("고정 snapshot 경로 불일치")
Path(sys.argv[3]).write_text(model_dir, encoding="utf-8")
"""
try:
    run_logged("model-download", [PYTHON, "-c", download_code, MODEL_ID, REVISION,
                                 str(WORK / "model-path.txt")], env=download_env)
finally:
    download_env.pop("HF_TOKEN", None)
    token = None
MODEL_DIR = (WORK / "model-path.txt").read_text(encoding="utf-8")
write_json(RESULTS / "model.json", {"id": MODEL_ID, "revision": REVISION, "path": MODEL_DIR})

## 묶음별 추론 — 끝난 묶음은 건너뜁니다

운영 `script.py` 를 `--debug-responses` 로 돌려 원응답을 남깁니다. 경로는 서버처럼 `PPS_*` 로 넘깁니다.
한 묶음이 성공하면 Drive 에 복사하고 `DONE.json` 을 씁니다.

모델이 한 공고에서 재시도까지 실패하면 `script.py` 는 실행 전체를 멈춥니다(실패를 0 으로 채우지 않는 계약).
그때는 그 공고 ID 를 `failed.json` 에 적고 **그 공고만 빼고** 묶음을 다시 돌립니다(최대 3회).
빠진 공고는 실패 집합 F 로 따로 보고합니다. 공고 ID 없이 실패하면(메모리 부족 등) 멈추고 로그를 받으세요.

In [ ]:
import re

FAILED_PATH = OUT_ROOT / "failed.json"
failed = json.loads(FAILED_PATH.read_text(encoding="utf-8")) if FAILED_PATH.exists() else {}
# 설계의 중단선: 실패 집합 F 가 표본의 1% 를 넘으면 원인부터 본다. 그 표본으로 D 를 만들지 않는다.
MAX_FAILED = N_NOTICES // 100
base_env = {key: value for key, value in os.environ.items()
            if not key.startswith(("PPS_", "VLLM_")) and key not in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN")}
base_env.update(PPS_MODEL_DIR=MODEL_DIR, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                PYTHONUNBUFFERED="1", CUDA_VISIBLE_DEVICES="0", VLLM_NO_USAGE_STATS="1",
                HF_HUB_DISABLE_TELEMETRY="1")

def prepare_case(run_name, ids):
    case = WORK / "cases" / run_name
    data = case / "data"
    data.mkdir(parents=True)
    for filename in ("script.py", "requirements.txt"):
        shutil.copyfile(REPO / filename, case / filename)
    for filename in ("항목표.json", "정답스키마_디코딩.json"):
        shutil.copyfile(REPO / "open/data" / filename, data / filename)
    shutil.copytree(REPO / "open/data/법령패키지", data / "법령패키지")
    payload = "".join(lines[i] + "\n" for i in ids).encode("utf-8")
    with (data / "test.jsonl.gz").open("wb") as raw_file:
        with gzip.GzipFile(fileobj=raw_file, mode="wb", mtime=0) as compressed:
            compressed.write(payload)
    return case, data

def failed_ids(diagnostics):
    found = set()
    if diagnostics.is_file():
        for line in diagnostics.read_text(encoding="utf-8").splitlines():
            event = json.loads(line)
            if event.get("event") == "run_failed":
                found |= set(re.findall(r"id=(PPS-[A-Z]+-\d+)", event.get("error_message", "")))
    return found

if len(failed) > MAX_FAILED:
    raise RuntimeError(f"실패 집합 F {len(failed)}건이 중단선 {MAX_FAILED}건을 넘었습니다. 원인부터 봅니다.")
for name, shard_ids in SHARDS:
    done_path = OUT_ROOT / name / "DONE.json"
    if done_path.exists():
        done = json.loads(done_path.read_text(encoding="utf-8"))
        if done["commit"] != SOURCE_COMMIT or done["planned"] != shard_ids:
            raise RuntimeError(f"{name}: Drive 의 기존 결과가 다른 코드/표본입니다. 섞지 않습니다.")
        print(f"{name}: 이미 완료 ({done['count']}건) — 건너뜁니다")
        continue
    for attempt in range(1, 4):
        run_name = f"{name}-a{attempt}"
        todo = [i for i in shard_ids if i not in failed]
        case, data = prepare_case(run_name, todo)
        env = dict(base_env, PPS_DATA_DIR=str(data), PPS_OUTPUT_DIR=str(RESULTS / run_name))
        try:
            run_logged(run_name, [PYTHON, "script.py", "--debug-responses"], env=env, cwd=case)
        except RuntimeError:
            bad = failed_ids(RESULTS / run_name / "diagnostics.jsonl")
            if not bad:
                raise RuntimeError(f"{run_name}: 공고 ID 없는 실패입니다. 마지막 셀로 로그를 받아 전달하세요.")
            for identifier in bad:
                failed[identifier] = {"shard": name, "attempt": attempt}
            write_json(FAILED_PATH, failed)
            if len(failed) > MAX_FAILED:
                raise RuntimeError(f"실패 집합 F {len(failed)}건이 중단선 {MAX_FAILED}건을 넘었습니다. "
                                   "원인부터 봅니다. 마지막 셀로 로그를 받아 전달하세요.")
            print(f"{run_name}: {sorted(bad)} 재시도 실패 → 실패 집합 F 로 빼고 다시 돌립니다")
            continue
        report = json.loads((RESULTS / run_name / "run_report.json").read_text(encoding="utf-8"))
        settings = report["reproduction"]["settings"]
        if (report["mode"] != "live" or report["건수"] != len(todo)
                or report["model_success_count"] != len(todo) or not settings["debug_responses"]):
            raise RuntimeError(f"{run_name}: 실제 모델 성공 건수/모드/원응답 설정 불일치")
        shutil.copytree(RESULTS / run_name, OUT_ROOT / name / "output")
        write_json(OUT_ROOT / name / "ids.json", todo)
        write_json(done_path, {"commit": SOURCE_COMMIT, "shard": name, "planned": shard_ids,
                               "count": len(todo), "attempt": attempt,
                               "elapsed_seconds": report["전체_s"], "per_notice_s": report["건당_s"]})
        print(f"{name}: 완료 {len(todo)}건, 건당 {report['건당_s']}초 → Drive 저장")
        break
    else:
        raise RuntimeError(f"{name}: 3회 모두 실패했습니다. 마지막 셀로 로그를 받아 전달하세요.")

done_all = [json.loads((OUT_ROOT / name / "DONE.json").read_text(encoding="utf-8")) for name, _ in SHARDS]
completed = sum(d["count"] for d in done_all)
if completed + len(failed) != N_NOTICES:
    raise RuntimeError(f"성공 {completed} + 실패 집합 F {len(failed)} != 표본 {N_NOTICES} — 결과가 섞였거나 빠졌습니다.")
write_json(OUT_ROOT / "summary.json", {"commit": SOURCE_COMMIT, "notices": N_NOTICES,
    "completed": completed, "failed": sorted(failed)})
print(f"전체 완료: {completed}건 성공, 실패 집합 F {len(failed)}건")

## 결과 ZIP 다운로드 — 실패했어도 이 셀 실행

설치·실행 로그와 Drive 에 저장된 묶음 결과(원응답·CSV·보고서)를 묶습니다. 이 ZIP 을 전달하면
PC 에서 삭제 집합 D 를 계산합니다. ZIP 이 크면 Drive 의 `unlabeled-d-…` 폴더를 통째로 내려받아도 됩니다.

In [ ]:
from google.colab import files

archive_path = WORK / ("unlabeled-d-results-" + str(time.time_ns()) + ".zip")
with zipfile.ZipFile(archive_path, "x", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file():
            archive.write(path, "logs/" + str(path.relative_to(RESULTS)))
    if "OUT_ROOT" in globals() and OUT_ROOT.exists():
        for path in sorted(OUT_ROOT.rglob("*")):
            if path.is_file():
                archive.write(path, "drive/" + str(path.relative_to(OUT_ROOT)))
files.download(str(archive_path))